# FREUID Challenge — Diagnostic: Grad-CAM Re-Check + Score Distribution - V6

**Context:** the `is_digital` flag hypothesis was ruled out (LB moved 0.22090 -> 0.21941, noise-level).
OOF FREUID is still ~0.0005 vs public LB ~0.22 — a ~440x gap that face-occlusion (which fixed the
original face-swap shortcut) has not closed. Public test uses the *same 5 document types* as
training, so this isn't classic cross-domain failure — it looks more like the model is still
relying on a narrow, generator-specific artifact rather than genuine semantic/structural fraud
cues (exactly what the competition's "anti-fragility" framing warns about).

**This notebook does two things, both inference-only (safe post-freeze, no weights touched):**

1. **Grad-CAM on real public test images** (not training/val images) — the leaderboard-relevant
   distribution — stratified into most-confident-fraud, most-confident-genuine, and borderline
   (closest to 0.5) buckets, with a reference face bounding box drawn on each so we can see at a
   glance whether attention is still glued to the face region, has moved to some other narrow
   artifact (borders, compression blocks, corners), or is genuinely diffuse across document
   structure.
2. **Calibrated score distribution** on the same public test set — a well-calibrated model facing
   genuinely hard/uncertain cases should show real mass in the middle of [0,1]; a model leaning on
   a narrow shortcut tends to be overconfident and bimodal (clustered near 0 and 1) even on cases
   it's actually getting wrong.

Uses the `fold0_best.pth` checkpoint only (single model, for interpretable single-path gradients —
Grad-CAM doesn't have a clean ensemble analogue). Uses `is_digital=1`, matching the configuration
that's actually on the leaderboard.

**Output:** grid image(s) under `/kaggle/working/gradcam_diagnostic/` plus a score-distribution
histogram and summary stats printed inline.


## 1. Environment Setup

In [ ]:
import os
import time
import math
import random
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

os.environ['NO_ALBUMENTATIONS_UPDATE'] = '1'
os.environ['HF_HUB_OFFLINE']           = '1'
os.environ['TRANSFORMERS_OFFLINE']     = '1'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.optimize
import scipy.stats
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.isotonic import IsotonicRegression
from PIL import Image

warnings.filterwarnings('ignore')

SEED = 42

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(SEED)

# Set this to True to intentionally run on CPU (slow — fine for ~24 Grad-CAM images,
# not recommended for the full-test-set score histogram pass). Leave False for GPU.
FORCE_CPU = False

DEVICE = torch.device('cpu' if FORCE_CPU or not torch.cuda.is_available() else 'cuda')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'Compute capability: {torch.cuda.get_device_capability(0)}')
print(f'PyTorch: {torch.__version__}')
print(f'timm   : {timm.__version__}')

# Fail-fast GPU/PyTorch kernel compatibility check (see prior diagnostic notebook —
# catches a P100-vs-T4 mismatch here in ~1s instead of deep in a later cell).
if DEVICE.type == 'cuda':
    try:
        _probe = torch.nn.Conv2d(3, 8, kernel_size=3, padding=1).to(DEVICE)
        _dummy = torch.randn(1, 3, 32, 32, device=DEVICE)
        with torch.no_grad():
            _ = _probe(_dummy)
        del _probe, _dummy
        torch.cuda.empty_cache()
        print('GPU compatibility check: OK')
    except Exception as e:
        raise RuntimeError(
            'GPU compatibility check FAILED for the assigned GPU '
            f'({torch.cuda.get_device_name(0)}, capability {torch.cuda.get_device_capability(0)}). '
            'Go to Notebook Settings -> Accelerator and select "GPU T4 x2", then re-run. '
            f'Original error: {e}'
        ) from e

## 2. Config

In [ ]:
@dataclass
class CFG:
    DATA_DIR:          str   = '/kaggle/input/datasets/maheshwarmishra/freuid-data'
    WEIGHTS_PATH:      str   = ('/kaggle/input/models/timm/tf-efficientnet/'
                                'pytorch/tf-efficientnet-b4/1/'
                                'tf_efficientnet_b4_aa-818f208c.pth')
    OUTPUT_DIR:        str   = '/kaggle/working'
    CHECKPOINT_DIR:    str   = '/kaggle/working/checkpoints'
    GRADCAM_DIR:       str   = '/kaggle/working/gradcam_diagnostic'
    USE_FULL_DATA:     bool  = True
    TRAIN_LABELS_FILE: str   = ''
    TEST_IMG_SUBDIR:   str   = 'public_test'

    BACKBONE:          str   = 'tf_efficientnet_b4'
    PRETRAINED:        bool  = True
    IMG_SIZE:          int   = 320
    DROP_RATE:         float = 0.3
    USE_METADATA:      bool  = True
    N_DOC_TYPES:       int   = 256   # recomputed in Section 4
    DOC_EMB_DIM:       int   = 16

    BATCH_SIZE:        int   = 16
    NUM_WORKERS:       int   = 0
    PIN_MEMORY:        bool  = True
    AMP:               bool  = True
    CALIB_METHOD:      str   = 'temperature'

    N_TOP:             int   = 8   # samples per bucket (high / low / borderline)
    FACE_DETECT_MAX_SIDE: int = 640

    def __post_init__(self):
        if not self.TRAIN_LABELS_FILE:
            self.TRAIN_LABELS_FILE = (
                'train_labels.csv' if self.USE_FULL_DATA
                else 'train_sample_labels.csv'
            )


CFG = CFG()
Path(CFG.CHECKPOINT_DIR).mkdir(parents=True, exist_ok=True)
Path(CFG.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(CFG.GRADCAM_DIR).mkdir(parents=True, exist_ok=True)
print('Config loaded (inference + visualization only).')
print(f'  IMG_SIZE: {CFG.IMG_SIZE}  |  N_TOP per bucket: {CFG.N_TOP}')

## 3. Checkpoint Restore

Same pattern as the `is_digital` diagnostic notebook. Update `CKPT_SOURCE_CANDIDATES` to match your attached dataset if this is a fresh session.

In [ ]:
import shutil

CKPT_SOURCE_CANDIDATES = [
    '/kaggle/input/datasets/maheshwarmishra/v4fold0',
]

existing = list(Path(CFG.CHECKPOINT_DIR).glob('fold*_best.pth'))
if existing:
    print(f'CHECKPOINT_DIR already populated: {[p.name for p in existing]}')
else:
    copied = []
    for src_dir in CKPT_SOURCE_CANDIDATES:
        src_path = Path(src_dir)
        if not src_path.exists():
            continue
        for ckpt_file in src_path.glob('fold*_best.pth'):
            dest = Path(CFG.CHECKPOINT_DIR) / ckpt_file.name
            shutil.copy2(ckpt_file, dest)
            copied.append(ckpt_file.name)
    print(f'Checkpoints copied into {CFG.CHECKPOINT_DIR}: {copied}')
    if not copied:
        print('WARNING: no checkpoint files found. Check the Input sidebar and update '
              'CKPT_SOURCE_CANDIDATES above.')

print('Current contents:', sorted(os.listdir(CFG.CHECKPOINT_DIR)))

## 4. Data Loading

Same vocab-rebuild logic as before. Test set built with `is_digital=1` to match the configuration actually on the leaderboard.

In [ ]:
IMAGE_EXTENSIONS = ('.jpeg', '.jpg', '.png', '.webp', '.bmp')

def find_images_in_dir(directory: Path) -> List[Path]:
    files = []
    for ext in IMAGE_EXTENSIONS:
        files += list(directory.glob(f'*{ext}'))
    return sorted(files)


def find_images_robust(base_dir: Path, subdir: str) -> Tuple[Path, List[Path]]:
    flat = base_dir / subdir
    files = find_images_in_dir(flat) if flat.exists() else []
    if files:
        return flat, files
    doubled = base_dir / subdir / subdir
    files = find_images_in_dir(doubled) if doubled.exists() else []
    if files:
        return doubled, files
    if flat.exists():
        for child in sorted(flat.iterdir()):
            if child.is_dir():
                files = find_images_in_dir(child)
                if files:
                    return child, files
    return flat, []


def resolve_image_path(base_dir: Path, rel_path: str) -> Path:
    c = base_dir / rel_path
    if c.exists(): return c
    parts = Path(rel_path).parts
    if len(parts) > 1:
        d = base_dir / parts[0] / rel_path
        if d.exists(): return d
    f = base_dir / Path(rel_path).name
    if f.exists(): return f
    return c


data_dir = Path(CFG.DATA_DIR)

labels_path = data_dir / CFG.TRAIN_LABELS_FILE
if not labels_path.exists():
    raise FileNotFoundError(f'Labels file not found: {labels_path}')
train_df = pd.read_csv(labels_path)
train_df['is_digital'] = train_df['is_digital'].astype(bool).astype(int)

all_types = train_df['type'].unique()
type2idx  = {t: i + 1 for i, t in enumerate(sorted(all_types))}
type2idx['<UNK>'] = 0
CFG.N_DOC_TYPES = len(type2idx) + 1
print(f'Recomputed N_DOC_TYPES: {CFG.N_DOC_TYPES}')

actual_test_dir, test_files = find_images_robust(data_dir, CFG.TEST_IMG_SUBDIR)
if not test_files:
    raise RuntimeError(f'No test images found under {data_dir / CFG.TEST_IMG_SUBDIR}')
rel_prefix = actual_test_dir.relative_to(data_dir)
print(f'Test images found: {len(test_files)} in {actual_test_dir}')

test_df = pd.DataFrame({
    'id':         [p.stem for p in test_files],
    'image_path': [str(rel_prefix / p.name) for p in test_files],
    'is_digital': 1,   # matches the configuration confirmed on the leaderboard
    'type_idx':   0,
})
print(f'test_df built: {len(test_df)} rows, is_digital=1 for all.')

## 5. Transforms

Single val transform only — no TTA needed here (Grad-CAM needs one clean forward/backward path per image; the score histogram uses a single model, single pass, which is representative enough for a distribution-shape check).

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def build_val_transform(img_size: int = CFG.IMG_SIZE) -> A.Compose:
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

VAL_TRANSFORM = build_val_transform()
print(f'Val transform defined. IMG_SIZE={CFG.IMG_SIZE}')

## 6. Dataset Class

In [ ]:
class FREUIDDataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path,
                 transform: Optional[A.Compose] = None) -> None:
        self.df        = df.reset_index(drop=True)
        self.data_dir  = data_dir
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def _resolve(self, rel_path: str) -> Path:
        c = self.data_dir / rel_path
        if c.exists(): return c
        parts = Path(rel_path).parts
        if len(parts) > 1:
            d = self.data_dir / parts[0] / rel_path
            if d.exists(): return d
        f = self.data_dir / Path(rel_path).name
        if f.exists(): return f
        return c

    def __getitem__(self, idx: int) -> Dict:
        row = self.df.iloc[idx]
        img_path = self._resolve(row['image_path'])
        try:
            image = np.array(Image.open(img_path).convert('RGB'))
        except Exception as e:
            print(f'Warning: cannot load {img_path}: {e}')
            image = np.zeros((CFG.IMG_SIZE, CFG.IMG_SIZE, 3), dtype=np.uint8)
        if self.transform is not None:
            image = self.transform(image=image)['image']
        return {
            'image':      image,
            'is_digital': torch.tensor(float(row.get('is_digital', 0)), dtype=torch.float32),
            'type_idx':   torch.tensor(int(row.get('type_idx', 0)), dtype=torch.long),
            'id':         str(row['id']),
        }

print('FREUIDDataset defined.')

## 7. Model Architecture (identical to v4)

In [ ]:
class FREUIDModel(nn.Module):
    def __init__(
        self,
        backbone_name: str   = CFG.BACKBONE,
        pretrained:    bool  = CFG.PRETRAINED,
        weights_path:  str   = CFG.WEIGHTS_PATH,
        n_doc_types:   int   = CFG.N_DOC_TYPES,
        doc_emb_dim:   int   = CFG.DOC_EMB_DIM,
        drop_rate:     float = CFG.DROP_RATE,
        use_metadata:  bool  = CFG.USE_METADATA,
    ) -> None:
        super().__init__()
        self.use_metadata = use_metadata

        self.backbone = timm.create_model(
            backbone_name, pretrained=False, num_classes=0, global_pool='avg')

        if pretrained:
            wp = Path(weights_path)
            if wp.exists():
                state_dict = torch.load(wp, map_location='cpu', weights_only=False)
                self.backbone.load_state_dict(state_dict, strict=False)
                print(f'  Backbone ImageNet weights loaded: {wp.name} (overwritten below)')

        feat_dim = self.backbone.num_features

        if use_metadata:
            self.doc_embedding = nn.Embedding(
                num_embeddings=n_doc_types, embedding_dim=doc_emb_dim, padding_idx=0)
            meta_dim = doc_emb_dim + 1
        else:
            meta_dim = 0

        in_dim    = feat_dim + meta_dim
        self.head = nn.Sequential(
            nn.LayerNorm(in_dim), nn.Dropout(drop_rate),
            nn.Linear(in_dim, 256), nn.GELU(),
            nn.Dropout(drop_rate / 2), nn.Linear(256, 1),
        )

    def forward(self, image: torch.Tensor,
                is_digital: torch.Tensor,
                type_idx: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(image)
        if self.use_metadata:
            feats = torch.cat(
                [feats, self.doc_embedding(type_idx), is_digital.unsqueeze(1)], dim=1)
        return self.head(feats).squeeze(1)

print('FREUIDModel defined.')

## 8. Calibration Classes

In [ ]:
class TemperatureScaler:
    def __init__(self): self.temperature = 1.0

    def fit(self, logits: np.ndarray, labels: np.ndarray) -> 'TemperatureScaler':
        def nll(t):
            t = float(t[0])
            if t <= 0: return 1e9
            p = np.clip(1.0 / (1.0 + np.exp(-logits / t)), 1e-7, 1 - 1e-7)
            return -np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p))
        res = scipy.optimize.minimize(nll, [1.0], method='L-BFGS-B', bounds=[(0.05, 20.0)])
        self.temperature = float(res.x[0])
        print(f'  Temperature = {self.temperature:.4f}')
        return self

    def transform(self, logits: np.ndarray) -> np.ndarray:
        return (1.0 / (1.0 + np.exp(-logits / self.temperature))).astype(np.float32)


class IsotonicCalibrator:
    def __init__(self): self.iso = IsotonicRegression(out_of_bounds='clip')

    def fit(self, logits: np.ndarray, labels: np.ndarray) -> 'IsotonicCalibrator':
        scores = 1.0 / (1.0 + np.exp(-logits))
        self.iso.fit(scores, labels)
        return self

    def transform(self, logits: np.ndarray) -> np.ndarray:
        scores = 1.0 / (1.0 + np.exp(-logits))
        return self.iso.predict(scores).astype(np.float32)


def get_calibrator():
    if CFG.CALIB_METHOD == 'temperature': return TemperatureScaler()
    if CFG.CALIB_METHOD == 'isotonic':    return IsotonicCalibrator()
    raise ValueError(f'Unknown calibration method: {CFG.CALIB_METHOD}')

print('Calibration classes defined.')

## 9. Load fold0 Checkpoint + Calibrator

Single model only — Grad-CAM needs one clean gradient path, and an ensemble doesn't have a natural single heatmap. fold0 was the stronger/longer-trained of the two (epoch 17 vs epoch 8, FREUID 0.0003 vs 0.0007).

In [ ]:
fold0_ckpt_path = Path(CFG.CHECKPOINT_DIR) / 'fold0_best.pth'
if not fold0_ckpt_path.exists():
    raise RuntimeError(f'{fold0_ckpt_path} not found. Check Section 3 (Checkpoint Restore).')

ckpt = torch.load(fold0_ckpt_path, map_location=DEVICE, weights_only=False)
model = FREUIDModel().to(DEVICE)
model.load_state_dict(ckpt['model_state'])
model.eval()

calibrator = get_calibrator()
calibrator.fit(ckpt['val_logits'], ckpt['val_labels'])

print(f'fold0 loaded — checkpoint FREUID (stored): {ckpt["val_metrics"]["freuid"]:.4f} '
      f'at epoch {ckpt["epoch"]}')

## 10. Score the Full Public Test Set (single model, no TTA)

Gives us (a) the score distribution for the calibration-shape check, and (b) the ranking used to pick Grad-CAM example buckets below.

In [ ]:

@torch.no_grad()
def score_dataset(model: nn.Module, df: pd.DataFrame, calibrator) -> Tuple[np.ndarray, List[str]]:
    ds = FREUIDDataset(df, data_dir, VAL_TRANSFORM)
    loader = DataLoader(ds, batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
                         num_workers=CFG.NUM_WORKERS, pin_memory=CFG.PIN_MEMORY)
    all_logits, all_ids = [], []
    for batch in loader:
        with torch.cuda.amp.autocast(enabled=CFG.AMP):
            out = model(batch['image'].to(DEVICE),
                        batch['is_digital'].to(DEVICE),
                        batch['type_idx'].to(DEVICE))
        all_logits.append(out.cpu().float().numpy())
        all_ids.extend(batch['id'])
    logits = np.concatenate(all_logits)
    scores = calibrator.transform(logits)
    return scores, all_ids


t0 = time.time()
test_scores, test_ids = score_dataset(model, test_df, calibrator)
print(f'Scored {len(test_scores)} test images in {time.time()-t0:.0f}s')

id_to_score = dict(zip(test_ids, test_scores))
test_df['score'] = test_df['id'].map(id_to_score)

print()
print('=== Score distribution summary (public test, fold0 only, is_digital=1) ===')
print(f'mean   : {test_scores.mean():.4f}')
print(f'std    : {test_scores.std():.4f}')
print(f'median : {np.median(test_scores):.4f}')
print(f'skew   : {scipy.stats.skew(test_scores):.4f}')
print(f'% < 0.1  (confident genuine): {(test_scores < 0.1).mean():.1%}')
print(f'% > 0.9  (confident fraud)  : {(test_scores > 0.9).mean():.1%}')
print(f'% in [0.4, 0.6] (uncertain) : {((test_scores >= 0.4) & (test_scores <= 0.6)).mean():.1%}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(test_scores, bins=50, color='#3b6ba5', edgecolor='none')
ax.set_xlabel('Calibrated score')
ax.set_ylabel('Count')
ax.set_title('Public test score distribution (fold0, is_digital=1)')
ax.axvline(0.5, color='red', ls='--', alpha=0.5, label='0.5')
ax.legend()
plt.tight_layout()
plt.savefig(f'{CFG.GRADCAM_DIR}/score_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print()
print('Reading guide: heavy mass piled near 0 and 1 (a "U-shape") with very little in the middle')
print('is consistent with an overconfident model leaning on a narrow shortcut/fingerprint rather')
print('than genuine, calibrated uncertainty about hard cases. Real spread through the middle is')
print('more consistent with the model actually grappling with ambiguous evidence.')


## 11. Face Bounding-Box Detection (reference overlay only)

Same Haar-cascade approach as v4's training-time face occlusion — applied here only to the handful of Grad-CAM sample images, purely so we can draw a reference box and visually judge overlap with the heatmap.

In [ ]:
_face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def detect_face_bbox(image_path: Path, max_side: int = CFG.FACE_DETECT_MAX_SIDE):
    try:
        img = cv2.imread(str(image_path))
        if img is None:
            return None
        h0, w0 = img.shape[:2]
        scale = max_side / max(h0, w0) if max(h0, w0) > max_side else 1.0
        small = cv2.resize(img, (int(w0 * scale), int(h0 * scale))) if scale != 1.0 else img
        gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
        faces = _face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4)
        if len(faces) == 0:
            return None
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
        inv = 1.0 / scale
        return (int(x * inv), int(y * inv), int(w * inv), int(h * inv))
    except Exception:
        return None

print('detect_face_bbox defined (Haar cascade, reference overlay only).')

## 12. Grad-CAM Implementation

Hooks `backbone.conv_head` — the last spatial feature map before global pooling in the EfficientNet-B4 backbone — via a forward hook (activations) and a full backward hook (gradients). Standard Grad-CAM: channel-wise gradient-weighted sum of activations, ReLU'd, upsampled to image resolution.

In [ ]:
class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, image: torch.Tensor, is_digital: torch.Tensor,
                 type_idx: torch.Tensor) -> Tuple[np.ndarray, float]:
        """image: (1, C, H, W), already on DEVICE. Returns (cam_HxW_in_[0,1], raw_logit)."""
        self.model.zero_grad(set_to_none=True)
        image = image.clone().requires_grad_(True)
        with torch.enable_grad():
            logit = self.model(image, is_digital, type_idx)
            logit.sum().backward()

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)          # (1, C, 1, 1)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)      # (1, 1, h, w)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=image.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze().float().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, float(logit.detach().cpu().item())


gradcam = GradCAM(model, model.backbone.conv_head)
print('GradCAM wired to model.backbone.conv_head.')

## 13. EXECUTION — Select Buckets, Run Grad-CAM, Save Overlay Grid

Three buckets, `CFG.N_TOP` images each: **most confident fraud** (highest score), **most confident genuine** (lowest score), and **borderline** (closest to 0.5 — often the most diagnostic, since these are exactly the cases the FREUID metric's 1% BPCER operating point is sensitive to).

In [ ]:

def get_overlay(image_rgb_uint8: np.ndarray, cam: np.ndarray) -> np.ndarray:
    """Resize cam to image size, apply a colormap, alpha-blend over the original image."""
    h, w = image_rgb_uint8.shape[:2]
    cam_resized = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap((cam_resized * 255).astype(np.uint8), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay = (0.55 * image_rgb_uint8 + 0.45 * heatmap).astype(np.uint8)
    return overlay


sorted_df = test_df.sort_values('score').reset_index(drop=True)
n = CFG.N_TOP

bucket_low        = sorted_df.head(n)                                                    # most confident genuine
bucket_high       = sorted_df.tail(n)                                                     # most confident fraud
bucket_borderline = sorted_df.iloc[(sorted_df['score'] - 0.5).abs().argsort()[:n]]        # closest to 0.5

buckets = {
    'most_confident_GENUINE (lowest score)': bucket_low,
    'most_confident_FRAUD (highest score)':   bucket_high,
    'BORDERLINE (closest to 0.5)':            bucket_borderline,
}

for bucket_name, bucket_df in buckets.items():
    safe_name = bucket_name.split(' ')[0]
    fig, axes = plt.subplots(2, n // 2 if n % 2 == 0 else n, figsize=(4 * (n // 2), 8))
    axes = np.array(axes).flatten()

    for ax, (_, row) in zip(axes, bucket_df.iterrows()):
        img_path = resolve_image_path(data_dir, row['image_path'])
        image_rgb = np.array(Image.open(img_path).convert('RGB'))

        tfm_out = VAL_TRANSFORM(image=image_rgb)['image'].unsqueeze(0).to(DEVICE)
        dig  = torch.tensor([float(row['is_digital'])], device=DEVICE)
        tidx = torch.tensor([int(row['type_idx'])], dtype=torch.long, device=DEVICE)

        cam, logit = gradcam.generate(tfm_out, dig, tidx)

        # Resize original image to model input size for overlay alignment
        image_resized = cv2.resize(image_rgb, (CFG.IMG_SIZE, CFG.IMG_SIZE))
        overlay = get_overlay(image_resized, cam)

        bbox = detect_face_bbox(img_path)
        if bbox is not None:
            h0, w0 = image_rgb.shape[:2]
            sx, sy = CFG.IMG_SIZE / w0, CFG.IMG_SIZE / h0
            x, y, w, h = bbox
            x0, y0 = int(x * sx), int(y * sy)
            x1, y1 = int((x + w) * sx), int((y + h) * sy)
            cv2.rectangle(overlay, (x0, y0), (x1, y1), (255, 255, 255), 2)

        ax.imshow(overlay)
        ax.set_title(f"score={row['score']:.3f}\nface_box={'yes' if bbox else 'no'}", fontsize=8)
        ax.axis('off')

    for ax in axes[len(bucket_df):]:
        ax.axis('off')

    fig.suptitle(f'Grad-CAM — {bucket_name}  (white box = detected face region, if any)',
                 fontsize=11)
    plt.tight_layout()
    out_path = f'{CFG.GRADCAM_DIR}/gradcam_{safe_name}.png'
    plt.savefig(out_path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'Saved: {out_path}')

print()
print('=' * 70)
print('READING GUIDE')
print('=' * 70)
print('- Heat concentrated tightly ON the white face box again -> old shortcut persists,')
print('  face-occlusion augmentation did not fully break it.')
print('- Heat concentrated on some OTHER narrow region (borders, corners, a fixed patch,')
print('  compression-block-looking areas) REGARDLESS of face position -> a new narrow')
print('  shortcut/fingerprint, likely tied to how frauds were synthesized in training data.')
print('- Heat spread across document structure (text fields, photo/background boundary,')
print('  security-feature areas) and DIFFERENT regions across different images -> genuinely')
print('  semantic, structural attention — a good sign, gap may be more about calibration/')
print('  distribution shift than shortcut reliance.')
print('- Compare BORDERLINE bucket especially closely -> these are the cases nearest the')
print('  decision boundary and most representative of where FREUID @ 1% BPCER will bite.')


In [ ]:
ct_counts = pd.crosstab(train_df['type'], train_df['label'])
ct_rates  = pd.crosstab(train_df['type'], train_df['label'], normalize='index')
ct_rates.columns = ['genuine_rate', 'fraud_rate']

print(ct_counts)
print()
print(ct_rates.round(3))